In [ ]:
# Llama-3.2 is gated. On Kaggle, store a secret named HF_TOKEN.
# Locally, set the HF_TOKEN environment variable or run `hf auth login` once.
import os
from huggingface_hub import login

try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret('HF_TOKEN')
except Exception:
    token = os.environ.get('HF_TOKEN')

try:
    if token:
        login(token=token)
        print('Hugging Face login: OK')
    else:
        print('No HF_TOKEN found; using the cached Hugging Face login if present.')
except Exception as exc:
    print('HF login was not completed automatically:', exc)
    print('If the model is already cached and accessible, you can continue.')


In [ ]:
import torch
from transformers import AutoModelForCausalLM
import matplotlib.pyplot as plt

model = AutoModelForCausalLM.from_pretrained('meta-llama/Llama-3.2-1B',)

In [ ]:
# TODO for Question 4
"""

Hint:
# Analyze the weights of the query projection layer (q_proj) where layer = k
 q_weight = model.model.layers[k].self_attn.q_proj.weight.detach()

# Apply SVD to decompose the weight matrix into U (left singular vectors), S (singular values), and V (right singular vectors).
 U, S, Vh = torch.linalg.svd(q_weight.float(), full_matrices=False)

# Compute the retained energy ratio (Cumulative energy) of each singular value.
energy_ratios = torch.cumsum(S ** 2, dim=0) / torch.sum(S ** 2) * 100
"""

LAYERS = [0, 7, 15]
ENERGY_THRESHOLD = 50.0   # percent
N_SHOW = 200              # only the first 200 singular values are plotted

q4_results = {}
for k in LAYERS:
    q_weight = model.model.layers[k].self_attn.q_proj.weight.detach()
    U, S, Vh = torch.linalg.svd(q_weight.float(), full_matrices=False)
    energy_ratios = torch.cumsum(S ** 2, dim=0) / torch.sum(S ** 2) * 100

    # Smallest r such that the top-r singular values keep >= 50% of the energy.
    min_rank = int(torch.searchsorted(energy_ratios, torch.tensor(ENERGY_THRESHOLD)).item()) + 1
    assert energy_ratios[min_rank - 1] >= ENERGY_THRESHOLD
    assert min_rank == 1 or energy_ratios[min_rank - 2] < ENERGY_THRESHOLD

    q4_results[k] = {'S': S, 'energy': energy_ratios, 'min_rank': min_rank}
    print(f'Layer {k:2d}: shape={tuple(q_weight.shape)}, full rank={len(S)}, '
          f'sigma_1={S[0]:.2f}, sigma_200={S[N_SHOW - 1]:.3f}, '
          f'min rank for >= {ENERGY_THRESHOLD:.0f}% energy = {min_rank} '
          f'({energy_ratios[min_rank - 1]:.2f}%)')

import pandas as pd
q4_table = pd.DataFrame(
    {f'Layer {k}': [q4_results[k]['min_rank']] for k in LAYERS},
    index=['Minimum Target Rank for query projection weights'],
)
q4_table


In [ ]:
from pathlib import Path

FIG_DIR = Path('outputs') / 'q4'
FIG_DIR.mkdir(parents=True, exist_ok=True)

SERIES = '#2a78d6'    # one series per chart -> one hue
INK = '#52514e'       # axes / secondary text
GRID = '#e4e3df'

plt.rcParams.update({
    'axes.edgecolor': INK, 'axes.labelcolor': INK,
    'xtick.color': INK, 'ytick.color': INK,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.color': GRID, 'grid.linewidth': 0.8,
    'axes.axisbelow': True, 'font.size': 10,
})

idx = list(range(1, N_SHOW + 1))
for k in LAYERS:
    r = q4_results[k]
    S_show = r['S'][:N_SHOW].numpy()
    E_show = r['energy'][:N_SHOW].numpy()

    # Singular value magnitudes (bar chart)
    fig, ax = plt.subplots(figsize=(6.4, 3.2))
    ax.bar(idx, S_show, width=0.8, color=SERIES, linewidth=0)
    ax.set_title(f'Layer {k} q_proj: Singular Value Magnitude (first {N_SHOW})', loc='left')
    ax.set_xlabel('Singular Value Index')
    ax.set_ylabel('Singular Value Magnitude')
    ax.set_xlim(0, N_SHOW + 1)
    ax.grid(axis='x', visible=False)
    fig.tight_layout()
    fig.savefig(FIG_DIR / f'layer{k}_singular_values.png', dpi=200)
    plt.show()

    # Cumulative energy (line chart) with the 50% threshold and the minimum rank marked
    fig, ax = plt.subplots(figsize=(6.4, 3.2))
    ax.plot(idx, E_show, color=SERIES, linewidth=2)
    ax.axhline(ENERGY_THRESHOLD, color=INK, linestyle='--', linewidth=1)
    m = r['min_rank']
    if m <= N_SHOW:
        ax.plot([m], [E_show[m - 1]], marker='o', markersize=6, color=SERIES,
                markeredgecolor='white', markeredgewidth=1.5)
        ax.annotate(f'rank {m}: {E_show[m - 1]:.1f}%', xy=(m, E_show[m - 1]),
                    xytext=(10, -16), textcoords='offset points', color=INK)
    ax.set_title(f'Layer {k} q_proj: Cumulative Energy (first {N_SHOW})', loc='left')
    ax.set_xlabel('Target Rank')
    ax.set_ylabel('Cumulative Energy (%)')
    ax.set_xlim(0, N_SHOW + 1)
    ax.set_ylim(0, 100)
    fig.tight_layout()
    fig.savefig(FIG_DIR / f'layer{k}_cumulative_energy.png', dpi=200)
    plt.show()

print('Figures saved to', FIG_DIR.resolve())
